In [1]:
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import pandas as pd
import numpy as np
import torch
import glob
import math
import sys

rc = {
    "axes.axisbelow": True,
    "axes.linewidth": 1.0,
    "axes.labelsize": "x-small",
    "axes.titlesize": "x-small",
    "figure.dpi": 150,
    "figure.figsize": (3.2, 3.2),
    "font.size": 12.0,
    "legend.fontsize": "xx-small",
    "lines.linewidth": 1.0,
    "savefig.transparent": True,
    "xtick.labelsize": "xx-small",
    "xtick.major.width": 1.0,
    "ytick.labelsize": "xx-small",
    "ytick.major.width": 1.0,
    "font.family": ["serif"],
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsfonts}",
}

plt.rcParams.update(rc)

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [2]:
def rgb_to_hex(r, g, b):
    return f"#{r:02x}{g:02x}{b:02x}"

MODEL_REGISTRY = {
    # 1-scale WFM models
    "WaveletFlow_1scale_haar_v4.0": dict(
        label  = "WFM 1-scale haar (ours)",
        #color  = rgb_to_hex(55, 82, 64),
        color  = rgb_to_hex(98, 124, 107),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "1-scale"},
        legend_rank = 0,
    ),
    "WaveletFlow_1scale_db2_v4.0": dict(
        label  = "WFM 1-scale db2 (ours)",
        #color  = rgb_to_hex(55, 82, 64),
        color  = rgb_to_hex(98, 124, 107),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "1-scale"},
        legend_rank = 0,
    ),
    "WaveletFlow_1scale_db4_v4.0": dict(
        label  = "WFM 1-scale db4 (ours)",
        #color  = rgb_to_hex(55, 82, 64),
        color  = rgb_to_hex(98, 124, 107),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "1-scale"},
        legend_rank = 0,
    ),
    "WaveletFlow_1scale_db6_v4.0": dict(
        label  = "WFM 1-scale db6 (ours)",
        #color  = rgb_to_hex(55, 82, 64),
        color  = rgb_to_hex(98, 124, 107),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "1-scale"},
        legend_rank = 0,
    ),
    # 3-scale WFM models
    "WaveletFlow_3scales_haar_v4.0": dict(
        label  = "WFM 3-scale haar (ours)",
        color  = rgb_to_hex(117, 176, 138),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "3-scale"},
        legend_rank = 1,
    ),
    "WaveletFlow_3scales_db2_v4.0": dict(
        label  = "WFM 3-scale db2 (ours)",
        color  = rgb_to_hex(117, 176, 138),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "3-scale"},
        legend_rank = 1,
    ),
    "WaveletFlow_3scales_db4_v4.0": dict(
        label  = "WFM 3-scale db4 (ours)",
        color  = rgb_to_hex(117, 176, 138),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "3-scale"},
        legend_rank = 1,
    ),
    "WaveletFlow_3scales_db6_v4.0": dict(
        label  = "WFM 3-scale db6 (ours)",
        color  = rgb_to_hex(117, 176, 138),
        dash   = (1, 0),
        lw     = 2.5,                    # ours: thicker
        groups = {"ours", "3-scale"},
        legend_rank = 1,
    ),
    "FM_data_space_v4.0": dict(
        label  = r"FM$_{pixel}$",
        color  = rgb_to_hex(176, 176, 176),
        dash   = (1, 0),
        lw     = 1.8,
        groups = {"generative"},
        legend_rank = 2,
    ),
    "FourierFlow_32modes_v4.0": dict(
        label  = r"FM$_{fourier}$",
        color  = rgb_to_hex(143, 94, 106),
        dash   = (1, 0),
        lw     = 1.8,
        groups = {"generative"},
        legend_rank = 3,
    ),
    "WDNO": dict(
        label  = "WDNO",
        color  = rgb_to_hex(191, 134, 148),
        dash   = (1, 0),
        lw     = 1.8,
        groups = {"generative"},
        legend_rank = 4,
    ),
    "WNO_3scales_db2": dict(
        label  = "WNO",
        color  = rgb_to_hex(176, 136, 84),
        #color  = rgb_to_hex(145, 74, 38),
        dash   = (1, 0),
        lw     = 1.8,
        groups = {"deterministic"},
        legend_rank = 5,
    ),
    "FNO": dict(
        label  = "FNO",
        #color  = rgb_to_hex(205, 170, 85),
        color  = rgb_to_hex(208, 169, 110),
        dash   = (1, 0),
        lw     = 1.8,
        groups = {"deterministic"},
        legend_rank = 6,
    ),
    "TFNO": dict(
        label  = "TFNO",
        #color  = rgb_to_hex(207, 185, 122),
        color  = rgb_to_hex(236, 227, 163),
        dash   = (1, 0),
        lw     = 1.8,
        groups = {"deterministic"},
        legend_rank = 7,
    ),
}

In [3]:
PDEs = ['turbulent_radiative_layer_2D', 'rayleigh_benard', 'active_matter']

#pde_best_model = {
#    "turbulent_radiative_layer_2D": ["WaveletFlow_1scale_db2_v4.0", "WaveletFlow_3scales_db2_v4.0"],
#    "rayleigh_benard":              ["WaveletFlow_1scale_db2_v4.0", "WaveletFlow_3scales_db4_v4.0"],
#    "active_matter":                ["WaveletFlow_1scale_db2_v4.0", "WaveletFlow_3scales_db2_v4.0"],
#}

constants = {'turbulent_radiative_layer_2D' : ["tcool"],
             'rayleigh_benard'              : ["Rayleigh", "Prandtl"],    
             'active_matter'                : ["zeta", "alpha"]}

model_checklist = [m for m in MODEL_REGISTRY]

palette     = {m: MODEL_REGISTRY[m]["color"] for m in model_checklist}
dashes      = {m: MODEL_REGISTRY[m]["dash"]  for m in model_checklist}
labels      = {m: MODEL_REGISTRY[m]["label"] for m in model_checklist}
linewidths  = {m: MODEL_REGISTRY[m]["lw"]    for m in model_checklist}

det_models  = [m for m in model_checklist if "deterministic" in MODEL_REGISTRY[m]["groups"]]
gen_models  = [m for m in model_checklist if "generative" in MODEL_REGISTRY[m]["groups"]]
#ours_models = [m for m in model_checklist if "ours" in MODEL_REGISTRY[m]["groups"]]
WFM_1scale_models = [m for m in model_checklist if "1-scale" in MODEL_REGISTRY[m]["groups"]]
WFM_3scale_models = [m for m in model_checklist if "3-scale" in MODEL_REGISTRY[m]["groups"]]
ours_models = ['WaveletFlow_1scale_haar_v4.0', 'WaveletFlow_1scale_db4_v4.0', 
               'WaveletFlow_3scales_db2_v4.0', 'WaveletFlow_3scales_db4_v4.0']

color_to_model = {v: k for k, v in palette.items()}

GT_LABEL             = "Ground-truth"
palette[GT_LABEL]    = "#222222"
dashes[GT_LABEL]     = (1, 0)
labels[GT_LABEL]     = "Ground-truth"
linewidths[GT_LABEL] = 1.8                                            

In [4]:
GT_LABEL = "Ground-truth"
LOAD_STATS = True
LOAD_PSD   = False

def enrich(df, stem, model_name, dataset_name):
    """Add metadata columns, preserving GT rows already labelled in the CSV."""
    parts     = stem.split('_')
    nsampling = next((p.replace('nsampling', '') for p in parts if p.startswith('nsampling')), None)
    nens      = next((p.replace('nens',      '') for p in parts if p.startswith('nens')),      None)
    if 'model' in df.columns:
        df.loc[df['model'] != GT_LABEL, 'model'] = model_name
    else:
        df['model'] = model_name
    df['source']    = stem
    df['dataset']   = dataset_name
    df['pde']       = dataset_name          # ← new: explicit PDE tag
    df['nsampling'] = int(nsampling) if nsampling else None
    df['nens']      = int(nens)      if nens      else None
    return df

dfs_stats, dfs_psd = [], []

for pde in PDEs:
    print(f"\n{'='*60}\nLoading PDE: {pde}\n{'='*60}")
    for model in model_checklist:
        pattern_stats = f'{PROJECT_ROOT}/experiments/{pde}/metrics/{model}*stats.csv'
        pattern_psd   = f'{PROJECT_ROOT}/experiments/{pde}/metrics/{model}*psd.csv'
        matched_files_stats = glob.glob(pattern_stats) if LOAD_STATS else []
        matched_files_psd   = glob.glob(pattern_psd) if LOAD_PSD else []

        if not matched_files_stats:
            print(f"  [WARN] No stats files for model '{model}' | pde '{pde}'")
        if not matched_files_psd:
            print(f"  [WARN] No psd   files for model '{model}' | pde '{pde}'")

        for fpath in sorted(matched_files_stats):
            try:
                df   = pd.read_csv(fpath)
                stem = Path(fpath).stem
                df   = enrich(df, stem, model, pde)
                dfs_stats.append(df)
                print(f"  [OK stats] {df.shape} | {Path(fpath).name:70s}")
            except Exception as e:
                print(f"  [ERR] {fpath}: {e}")

        for fpath in sorted(matched_files_psd):
            try:
                df   = pd.read_csv(fpath)
                stem = Path(fpath).stem
                df   = enrich(df, stem, model, pde)
                dfs_psd.append(df)
                print(f"  [OK psd  ] {df.shape} | {Path(fpath).name:70s}")
            except Exception as e:
                print(f"  [ERR] {fpath}: {e}")

# ── Concat & deduplicate ───────────────────────────────────────────────────────
DEDUP_EXCLUDE = {'source', 'nsampling', 'nens'}

if dfs_stats:
    stats_df = pd.concat(dfs_stats, ignore_index=True)
    stats_df = stats_df.drop_duplicates(
        subset=[c for c in stats_df.columns if c not in DEDUP_EXCLUDE]
    )
    print(f"\nstats_df: {stats_df.shape}  |  columns: {list(stats_df.columns)}")
else:
    stats_df = pd.DataFrame()
    print("[WARN] No data loaded — stats_df is empty.")

if dfs_psd:
    psd_df = pd.concat(dfs_psd, ignore_index=True)
    psd_df = psd_df.drop_duplicates(
        subset=[c for c in psd_df.columns if c not in DEDUP_EXCLUDE]
    )
    print(f"\npsd_df:   {psd_df.shape}  |  columns: {list(psd_df.columns)}")
else:
    psd_df = pd.DataFrame()
    print("[WARN] No data loaded — psd_df is empty.")


Loading PDE: turbulent_radiative_layer_2D
  [WARN] No psd   files for model 'WaveletFlow_1scale_haar_v4.0' | pde 'turbulent_radiative_layer_2D'
  [OK stats] (3492, 39) | WaveletFlow_1scale_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [WARN] No psd   files for model 'WaveletFlow_1scale_db2_v4.0' | pde 'turbulent_radiative_layer_2D'
  [OK stats] (3492, 39) | WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [WARN] No psd   files for model 'WaveletFlow_1scale_db4_v4.0' | pde 'turbulent_radiative_layer_2D'
  [OK stats] (3492, 39) | WaveletFlow_1scale_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [WARN] No psd   files for model 'WaveletFlow_1scale_db6_v4.0' | pde 'turbulent_radiative_layer_2D'
  [OK stats] (3492, 39) | WaveletFlow_1scale_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [WARN] No psd   files for model 'WaveletFlow_3scales_haar_v4.0' | pde 'turbulent_radiative_layer_2D'
  [O

# Config

In [5]:
PDE_LABELS = {
    'turbulent_radiative_layer_2D' : 'Turbulent Radiative Layer 2D',
    'rayleigh_benard'              : 'Rayleigh-Bénard Convection',
    'active_matter'                : 'Active Matter',
    }

# x-axis limits for PDE
pde_xlims = {
    'turbulent_radiative_layer_2D' : (-5, 100),
    'rayleigh_benard'              : (-5, 200),
    'active_matter'                : (-5, 80),
    }

band_ylims = {
    "Low"   : (-0.01, 0.5), 
    "Mid"   : (-0.01, 0.5), 
    "High"  : (-0.01, 0.5)
    }

bands      = ["Low", "Mid", "High"]

LEGEND_LW  = 1.8
TEXT_WIDTH = 14    # inches
ROW_HEIGHT = 2.8   # inches per band row

## Compute horizon and model_type attributes

In [6]:
aggregate_mode = "mean"

def horizon_rayleigh_benard(t):
    if t < 32:
        return "01:32"
    elif t < 64:
        return "33:64"
    else:
        return "65:196"

"""
def horizon_rayleigh_benard(t):
    if t < 20:
        return "01:20"
    elif t < 60:
        return "21:60"
    else:
        return "61:196"
"""

"""
def horizon_active_matter(t):
    if t < 24:
        return "01:24"
    elif t < 52:
        return "25:52"
    else:
        return "52:77"
"""

def horizon_active_matter(t):
    if t < 16:
        return "01:16"
    elif t < 32:
        return "17:32"
    else:
        return "33:77"
    
def horizon_turbulent_radiative_layer_2D(t):
    if t < 32:
        return "01:32"
    elif t < 64:
        return "33:64"
    else:
        return "65:97"

"""
def horizon_turbulent_radiative_layer_2D(t):
    if t < 20:
        return "01:20"
    elif t < 60:
        return "21:60"
    else:
        return "61:97"
"""

def aggregate(x):
    if "mean" in aggregate_mode:
        return np.nanmean(x)
    elif "median" in aggregate_mode:
        return np.nanmedian(x)

def model_type(model):
    if model in gen_models:
        return "generative"
    if model in WFM_1scale_models:
        return "1-scale"
    if model in WFM_3scale_models:
        return "3-scale"
    #if model in ours_models:
    #    return "ours"
    return "deterministic"

# ── Horizon: write into matching rows only, don't overwrite the full column ───
horizon_fn = {
    'turbulent_radiative_layer_2D': horizon_turbulent_radiative_layer_2D,
    'rayleigh_benard':              horizon_rayleigh_benard,
    'active_matter':                horizon_active_matter,
}

if LOAD_STATS:
    stats_df['step'] = stats_df['step'] + 1

    field_names = stats_df['field'].unique()
    n_fields = len(field_names)

    stats_df["horizon"] = np.nan                         
    for pde, fn in horizon_fn.items():
        mask = stats_df["pde"] == pde
        stats_df.loc[mask, "horizon"] = stats_df.loc[mask, "step"].apply(fn)
    stats_df["model_type"] = stats_df["model"].apply(model_type)
    stats_df["spread_skill_new"] = (stats_df["spread"] + 1e-3) / (stats_df["rmse"] + 1e-3)

if LOAD_PSD:
    psd_df['step']   = psd_df['step'] + 1

    field_names = psd_df['field'].unique()
    n_fields = len(field_names)
    
    psd_df["horizon"] = np.nan                         
    for pde, fn in horizon_fn.items():
        mask = psd_df["pde"] == pde
        psd_df.loc[mask, "horizon"] = psd_df.loc[mask, "step"].apply(fn)
    psd_df["model_type"] = psd_df["model"].apply(model_type)

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_28502/3656787083.py:90: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['01:32' '01:32' '01:32' ... '65:97' '65:97' '65:97']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats_df.loc[mask, "horizon"] = stats_df.loc[mask, "step"].apply(fn)


## Highlight helpers

In [7]:
def highlight_min_per_field_horizon(df: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    fields = df.index.get_level_values("field").unique()
    for field in fields:
        field_mask = df.index.get_level_values("field") == field
        sub = df.loc[field_mask]
        for col in df.columns:
            valid = sub[col].dropna()
            if valid.empty:
                continue
            sorted_vals = valid.nsmallest(2)
            if len(sorted_vals) >= 1:
                styles.loc[sorted_vals.index[0], col] = "font-weight: bold"
            if len(sorted_vals) >= 2:
                styles.loc[sorted_vals.index[1], col] = "text-decoration: underline"
    return styles


def highlight_min_per_horizon(df: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for col in df.columns:
        valid = df[col].dropna()
        if valid.empty:
            continue
        sorted_vals = valid.nsmallest(2)
        if len(sorted_vals) >= 1:
            styles.loc[sorted_vals.index[0], col] = "font-weight: bold"
        if len(sorted_vals) >= 2:
            styles.loc[sorted_vals.index[1], col] = "text-decoration: underline"
    return styles

# 1. Deterministic Scores

## 1.1. VRMSE vs Rollout: turbulent_radiative_layer_2D

In [8]:
per_field = False
metrics_to_show = ['vrmse']

PDE = 'turbulent_radiative_layer_2D'

if per_field:
    table = stats_df[stats_df["dataset"] == PDE][["model", "field", "horizon"] + metrics_to_show]
    table = table.groupby(["model", "field", "horizon"]).agg(aggregate).round(3).reset_index()
    table = table.pivot(index=["model", "field"], columns="horizon", values=metrics_to_show)
else:
    table = stats_df[stats_df["dataset"] == PDE][["model", "horizon"] + metrics_to_show]
    table = table.groupby(["model", "horizon"]).agg(aggregate).round(3).reset_index()
    table = table.pivot(index=["model"], columns="horizon", values=metrics_to_show)

#table.style.apply(highlight_min_per_field_horizon, axis=None).format("{:.3f}")  
table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_28502/3656787083.py:59: RuntimeWarning: Mean of empty slice
  return np.nanmean(x)


## 1.2. VRMSE vs Rollout: Rayleigh-Bénard

In [9]:
per_field = False
metrics_to_show = ['vrmse']

PDE = 'rayleigh_benard'

if per_field:
    table = stats_df[stats_df["dataset"] == PDE][["model", "field", "horizon"] + metrics_to_show]
    table = table.groupby(["model", "field", "horizon"]).agg(aggregate).round(3).reset_index()
    table = table.pivot(index=["model", "field"], columns="horizon", values=metrics_to_show)
else:
    table = stats_df[stats_df["dataset"] == PDE][["model", "horizon"] + metrics_to_show]
    table = table.groupby(["model", "horizon"]).agg(aggregate).round(3).reset_index()
    table = table.pivot(index=["model"], columns="horizon", values=metrics_to_show)

#table.style.apply(highlight_min_per_field_horizon, axis=None).format("{:.3f}")  
table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

## 1.3. VRMSE vs Rollout: Active Matter

In [10]:
per_field = False
metrics_to_show = ['vrmse']

PDE = 'active_matter'

if per_field:
    table = stats_df[stats_df["dataset"] == PDE][["model", "field", "horizon"] + metrics_to_show]
    table = table.groupby(["model", "field", "horizon"]).agg(aggregate).round(3).reset_index()
    table = table.pivot(index=["model", "field"], columns="horizon", values=metrics_to_show)
else:
    table = stats_df[stats_df["dataset"] == PDE][["model", "horizon"] + metrics_to_show]
    table = table.groupby(["model", "horizon"]).agg(aggregate).round(3).reset_index()
    table = table.pivot(index=["model"], columns="horizon", values=metrics_to_show)

#table.style.apply(highlight_min_per_field_horizon, axis=None).format("{:.3f}")  
table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

## 1.4. VRMSE Deterministic models for all PDEs

In [11]:
per_field = False
metrics_to_show = ['vrmse']

table = stats_df[stats_df["model_type"] == 'deterministic']

if per_field:
    table = (
        table[["dataset", "model", "field", "horizon"] + metrics_to_show]
        .groupby(["model", "field", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index=["model", "field"], columns=["dataset", "horizon"], values=metrics_to_show)
    )
else:
    table = (
        table[["dataset", "model", "horizon"] + metrics_to_show]
        .groupby(["model", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index="model", columns=["dataset", "horizon"], values=metrics_to_show)
    )

# Drop the top-level metric name if only one metric is shown
if len(metrics_to_show) == 1:
    table.columns = table.columns.droplevel(0)

table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_28502/3656787083.py:59: RuntimeWarning: Mean of empty slice
  return np.nanmean(x)


## 1.5. CRPS Generative models for all PDEs

In [12]:
per_field = False
metrics_to_show = ['crps']

table = stats_df[stats_df["model_type"] == 'generative']

if per_field:
    table = (
        table[["dataset", "model", "field", "horizon"] + metrics_to_show]
        .groupby(["model", "field", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index=["model", "field"], columns=["dataset", "horizon"], values=metrics_to_show)
    )
else:
    table = (
        table[["dataset", "model", "horizon"] + metrics_to_show]
        .groupby(["model", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index="model", columns=["dataset", "horizon"], values=metrics_to_show)
    )

# Drop the top-level metric name if only one metric is shown
if len(metrics_to_show) == 1:
    table.columns = table.columns.droplevel(0)

table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

## 1.6. VRMSE Deterministic + Generative models for all PDEs

In [13]:
per_field = False
metrics_to_show = ['vrmse']

table = stats_df[(stats_df["model_type"].isin(['generative', 'deterministic']))]

if per_field:
    table = (
        table[["dataset", "model", "field", "horizon"] + metrics_to_show]
        .groupby(["model", "field", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index=["model", "field"], columns=["dataset", "horizon"], values=metrics_to_show)
    )
else:
    table = (
        table[["dataset", "model", "horizon"] + metrics_to_show]
        .groupby(["model", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index="model", columns=["dataset", "horizon"], values=metrics_to_show)
    )

# Drop the top-level metric name if only one metric is shown
if len(metrics_to_show) == 1:
    table.columns = table.columns.droplevel(0)

table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_28502/3656787083.py:59: RuntimeWarning: Mean of empty slice
  return np.nanmean(x)


## 1.7. [*] CRPS Deterministic + Generative models for all PDEs

In [34]:
per_field = False
metrics_to_show = ['crps']
dataset_order = ["turbulent_radiative_layer_2D", "rayleigh_benard", "active_matter"]

#table = stats_df[stats_df["model_type"].isin(['generative', 'deterministic'])]
table = stats_df[stats_df["model"].isin(gen_models + det_models + ours_models)]

if per_field:
    table = (
        table[["dataset", "model", "field", "horizon"] + metrics_to_show]
        .groupby(["model", "field", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index=["model", "field"], columns=["dataset", "horizon"], values=metrics_to_show)
    )
else:
    table = (
        table[["dataset", "model", "horizon"] + metrics_to_show]
        .groupby(["model", "dataset", "horizon"])
        .agg(aggregate)
        .round(3)
        .reset_index()
        .pivot(index="model", columns=["dataset", "horizon"], values=metrics_to_show)
    )

# Reorder while columns are still (metric, dataset, horizon)
dataset_level = 1 if len(metrics_to_show) > 1 or True else 0
table = table.reindex(
    columns=pd.MultiIndex.from_tuples(
        [col for ds in dataset_order for col in table.columns if col[dataset_level] == ds]
    )
)

# Drop the top-level metric name if only one metric is shown
if len(metrics_to_show) == 1:
    table.columns = table.columns.droplevel(0)

# ── Styled display ────────────────────────────────────────────────────────────
table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")

# ── LaTeX with \textbf (best) and \underline (2nd best) per column ───────────
def latex_cell(val, rank):
    s = f"{val:.3f}"
    if rank == 1: return rf"\textbf{{{s}}}"
    if rank == 2: return rf"\underline{{{s}}}"
    return s

ranks = {}
for col in table.columns:
    sorted_idx = table[col].dropna().sort_values().index.tolist()
    ranks[col] = {m: sorted_idx.index(m) + 1 for m in sorted_idx}

data_rows = []
for model in table.index:
    cells = [labels.get(model, model)]
    for col in table.columns:
        val = table.loc[model, col]
        cells.append("--" if pd.isna(val) else latex_cell(val, ranks[col].get(model, 99)))
    data_rows.append(" & ".join(cells) + r" \\")

# Column header: (dataset, horizon) MultiIndex
dataset_labels = {
    "turbulent_radiative_layer_2D": "Turbulence",
    "rayleigh_benard":              "Rayleigh--Bénard",
    "active_matter":                "Active Matter",
}
datasets  = table.columns.get_level_values(0)
horizons  = table.columns.get_level_values(1)

ds_groups = {}
for i, (ds, hz) in enumerate(zip(datasets, horizons)):
    ds_groups.setdefault(ds, []).append(i)

# i+2 because LaTeX columns are 1-indexed and we have the Model column first
cmidrule = " ".join(
    rf"\cmidrule(lr){{{min(idxs) + 2}-{max(idxs) + 2}}}"
    for ds, idxs in ds_groups.items()
)
header_ds = "Model & " + " & ".join(
    rf"\multicolumn{{{len(idxs)}}}{{c}}{{{dataset_labels.get(ds, ds)}}}"
    for ds, idxs in ds_groups.items()
) + r" \\"
header_hz = "& " + " & ".join(str(hz) for hz in horizons) + r" \\"
n_cols     = len(table.columns)

latex = "\n".join([
    rf"\begin{{tabular}}{{l{'c' * n_cols}}}",
    r"\toprule",
    header_ds,
    cmidrule,
    header_hz,
    r"\midrule",
    *data_rows,
    r"\bottomrule",
    r"\end{tabular}",
])
print(latex)

\begin{tabular}{lccccccccc}
\toprule
Model & \multicolumn{3}{c}{Turbulence} & \multicolumn{3}{c}{Rayleigh--Bénard} & \multicolumn{3}{c}{Active Matter} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
& 01:32 & 33:64 & 65:97 & 01:32 & 33:64 & 65:196 & 01:16 & 17:32 & 33:77 \\
\midrule
FM$_{pixel}$ & \textbf{0.412} & 1.076 & 1.237 & \underline{0.004} & 0.034 & 0.078 & \textbf{0.054} & \textbf{0.162} & \textbf{0.194} \\
FNO & 2.380 & 7.168 & 10.406 & 0.025 & 0.072 & 30564885488571937174203662336.000 & 0.159 & 0.341 & 0.376 \\
FM$_{fourier}$ & 0.846 & 1.379 & 1.526 & 0.009 & 0.036 & 0.095 & 0.182 & 0.224 & 0.217 \\
TFNO & 1.995 & 5.798 & 5.770 & 0.235 & 0.269 & 0.326 & 0.140 & 0.356 & 0.382 \\
WDNO & 2.444 & 3.342 & 4.274 & 0.049 & 0.101 & 0.138 & 0.216 & 0.245 & 0.266 \\
WNO & 5266973324952344.000 & inf & -- & 0.076 & 0.160 & 0.215 & 0.336 & 0.462 & 0.480 \\
WFM 1-scale db4 (ours) & 0.523 & 1.067 & 1.227 & 0.007 & 0.039 & \underline{0.072} & 0.322 & 0.388 & 0.388 \\
WFM 1-scal

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_28502/3656787083.py:59: RuntimeWarning: Mean of empty slice
  return np.nanmean(x)


In [35]:
table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")

# 2. Spectral Scores

In [16]:
def highlight_min_per_pde(df):
    """Bold the minimum, underline the second minimum, within each PDE group."""
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for pde in df.index.get_level_values("pde").unique():
        mask = df.index.get_level_values("pde") == pde
        sub = df.loc[mask]
        for col in df.columns:
            sorted_vals = sub[col].dropna().sort_values()
            best   = sorted_vals.iloc[0] if len(sorted_vals) > 0 else None
            second = sorted_vals.iloc[1] if len(sorted_vals) > 1 else None
            styles.loc[mask, col] = sub[col].apply(
                lambda v: "font-weight: bold; color: black"      if v == best   else
                          "text-decoration: underline; color: black" if v == second else ""
            )
    return styles

## [*] 2.1. Aggregated Spectral Coherence RMSE for generative models across all PDEs and Bands

In [17]:
def highlight_min_per_col(df):
    """Bold best, underline 2nd best, per column."""
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for col in df.columns:
        vals = df[col].dropna()
        if len(vals) < 1: continue
        sorted_idx = vals.sort_values().index
        styles.loc[sorted_idx[0], col] = "font-weight: bold"
        if len(sorted_idx) > 1:
            styles.loc[sorted_idx[1], col] = "text-decoration: underline"
    return styles

# ── LaTeX with \textbf (best) and \underline (2nd best) per column ───────────
def latex_cell(val, rank):
    s = f"{val:.3f}"
    if rank == 1: return rf"\textbf{{{s}}}"
    if rank == 2: return rf"\underline{{{s}}}"
    return s

In [ ]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}
bands     = list(band_cols.values())

# ── Build table: index=model, columns=(pde, band) ────────────────────────────
table = (
    #stats_df[stats_df["model_type"].isin(["generative"])][
    #stats_df[(stats_df["model"].isin(ours_models)) & (stats_df["model_type"] == "generative")][
    stats_df[(stats_df["model_type"] == "generative") | (stats_df["model_type"] == "1-scale") | (stats_df["model_type"] == "3-scale")][   
    #stats_df[(stats_df["model_type"] == "generative") | (stats_df["model_type"] == "3-scale")][   
        ["pde", "model", "horizon"] + list(band_cols.keys())
    ]
    .groupby(["pde", "model", "horizon"])
    .agg(aggregate)
    .round(3)
    .reset_index()
    .rename(columns=band_cols)
    .groupby(["pde", "model"])[bands]
    .mean()
    .round(3)
    .reset_index()
    .pivot(index="model", columns="pde", values=bands)
    .swaplevel(axis=1)                          
    .reindex(columns=pd.MultiIndex.from_tuples(
        [(pde, band) for pde in PDEs for band in bands]
    ))
)

# ── Styled display ────────────────────────────────────────────────────────────
table.style.apply(highlight_min_per_col, axis=None).format("{:.3f}")

### Export to LaTeX

In [33]:
n_bands = len(bands)
n_pdes  = len(PDEs)
n_cols  = n_pdes * n_bands

# column ranks per (pde, band)
ranks = {}
for col in table.columns:
    sorted_idx = table[col].dropna().sort_values().index.tolist()
    ranks[col] = {m: sorted_idx.index(m) + 1 for m in sorted_idx}

# header lines
cmidrule = " ".join(
    rf"\cmidrule(lr){{{2 + i*n_bands}-{1 + (i+1)*n_bands}}}"
    for i in range(n_pdes)
)
header_pde  = "Model & " + " & ".join(
    rf"\multicolumn{{{n_bands}}}{{c}}{{{PDE_LABELS[p]}}}" for p in PDEs
) + r" \\"
header_band = "& " + " & ".join(bands * n_pdes) + r" \\"

# data rows
data_rows = []
for model in table.index:
    cells = [labels.get(model, model)]
    for col in table.columns:
        val = table.loc[model, col]
        cells.append("--" if pd.isna(val) else latex_cell(val, ranks[col].get(model, 99)))
    data_rows.append(" & ".join(cells) + r" \\")

latex = "\n".join([
    rf"\begin{{tabular}}{{l{'c' * n_cols}}}",
    r"\toprule",
    header_pde,
    cmidrule,
    header_band,
    r"\midrule",
    *data_rows,
    r"\bottomrule",
    r"\end{tabular}",
])
print(latex)

\begin{tabular}{lccccccccc}
\toprule
Model & \multicolumn{3}{c}{Turbulent Radiative Layer 2D} & \multicolumn{3}{c}{Rayleigh-Bénard Convection} & \multicolumn{3}{c}{Active Matter} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
& Low & Mid & High & Low & Mid & High & Low & Mid & High \\
\midrule
FM$_{pixel}$ & \textbf{0.033} & \textbf{0.018} & \textbf{0.504} \\
FM$_{fourier}$ & \underline{0.105} & \underline{0.046} & \underline{1.206} \\
WDNO & 0.178 & 0.110 & 3.530 \\
\bottomrule
\end{tabular}


## Aggregate across bands

In [20]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}
bands     = list(band_cols.values())

table = (
    #stats_df[stats_df["model_type"].isin(["generative"])][
    #stats_df[(stats_df["model"].isin(ours_models)) & (stats_df["model_type"] == "generative")][
    stats_df[(stats_df["model_type"] == "generative") | 
             (stats_df["model_type"] == "1-scale") | 
             (stats_df["model_type"] == "3-scale")][   
    #stats_df[(stats_df["model_type"] == "generative") | (stats_df["model_type"] == "3-scale")][   
        ["pde", "model", "horizon"] + list(band_cols.keys())
    ]
    .groupby(["pde", "model", "horizon"])
    .agg(aggregate)
    .round(3)
    .reset_index()
    .rename(columns=band_cols)
    .groupby(["pde", "model"])[bands]
    .mean()
    .round(3)
    .reset_index()
    .pivot(index="model", columns="pde", values=bands)
    .swaplevel(axis=1)                          # → (pde, band)
    .reindex(columns=pd.MultiIndex.from_tuples(
        [(pde, band) for pde in PDEs for band in bands]
    ))
)

# ── Aggregate across bands for each PDE ──────────────────────────────────────
table_by_pde = (
    table
    .T.groupby(level=0).mean().T   # mean over Low/Mid/High for each PDE
    .reindex(columns=PDEs)         # restore PDE column order
    .round(4)
)
table_by_pde.style.apply(highlight_min_per_col, axis=None).format("{:.3f}")

,turbulent_radiative_layer_2D,rayleigh_benard,active_matter
model,,,
FM_data_space_v4.0,0.152,0.131,0.241
FourierFlow_32modes_v4.0,0.379,0.264,0.375
WDNO,0.270,0.466,0.303
WaveletFlow_1scale_db2_v4.0,0.152,0.132,0.287
WaveletFlow_1scale_db4_v4.0,0.148,0.119,0.303
WaveletFlow_1scale_db6_v4.0,0.150,0.132,0.290
WaveletFlow_1scale_haar_v4.0,0.148,0.139,0.241
WaveletFlow_3scales_db2_v4.0,0.154,0.111,0.219
WaveletFlow_3scales_db4_v4.0,0.154,0.109,0.247


In [21]:
def to_latex_bold_underline(df):
    """Export df to LaTeX: bold = best (min), underline = second best per column."""
    
    col_vals  = df.values.astype(float)
    formatted = df.round(4).astype(str)

    rows = []
    for col_idx in range(df.shape[1]):
        col = col_vals[:, col_idx]
        valid = col[~np.isnan(col)]
        if len(valid) < 1:
            continue
        sorted_vals = np.sort(valid)
        best        = sorted_vals[0]
        second_best = sorted_vals[1] if len(sorted_vals) > 1 else None

        for row_idx in range(df.shape[0]):
            v = col[row_idx]
            s = f"{v:.3f}"
            if np.isnan(v):
                formatted.iloc[row_idx, col_idx] = "--"
            elif np.isclose(v, best):
                formatted.iloc[row_idx, col_idx] = r"\textbf{" + s + "}"
            elif second_best is not None and np.isclose(v, second_best):
                formatted.iloc[row_idx, col_idx] = r"\underline{" + s + "}"
            else:
                formatted.iloc[row_idx, col_idx] = s

    n_cols   = len(df.columns)
    col_spec = "l" + "c" * n_cols
    header   = " & ".join([df.index.name or "Model"] + list(df.columns)) + r" \\"

    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        header,
        r"\midrule",
    ]
    for model, row in formatted.iterrows():
        lines.append(model + " & " + " & ".join(row.values) + r" \\")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{Band-averaged SRMSE per PDE. \textbf{Bold}: best, \underline{underline}: second best.}",
        r"\label{tab:band_avg_srmse}",
        r"\end{table}",
    ]

    return "\n".join(lines)


latex_str = to_latex_bold_underline(table_by_pde)
print(latex_str)

# ── Optionally save to file ───────────────────────────────────────────────────
with open("table_by_pde.tex", "w") as f:
    f.write(latex_str)

\begin{table}[ht]
\centering
\begin{tabular}{lccc}
\toprule
model & turbulent_radiative_layer_2D & rayleigh_benard & active_matter \\
\midrule
FM_data_space_v4.0 & 0.152 & 0.131 & 0.241 \\
FourierFlow_32modes_v4.0 & 0.379 & 0.264 & 0.375 \\
WDNO & 0.270 & 0.466 & 0.303 \\
WaveletFlow_1scale_db2_v4.0 & 0.152 & 0.132 & 0.287 \\
WaveletFlow_1scale_db4_v4.0 & \textbf{0.148} & 0.119 & 0.303 \\
WaveletFlow_1scale_db6_v4.0 & 0.150 & 0.132 & 0.290 \\
WaveletFlow_1scale_haar_v4.0 & \textbf{0.148} & 0.139 & 0.241 \\
WaveletFlow_3scales_db2_v4.0 & 0.154 & \underline{0.111} & \textbf{0.219} \\
WaveletFlow_3scales_db4_v4.0 & 0.154 & \textbf{0.109} & 0.247 \\
WaveletFlow_3scales_db6_v4.0 & 0.152 & 0.112 & \underline{0.237} \\
WaveletFlow_3scales_haar_v4.0 & 0.148 & 0.131 & 0.245 \\
\bottomrule
\end{tabular}
\caption{Band-averaged SRMSE per PDE. \textbf{Bold}: best, \underline{underline}: second best.}
\label{tab:band_avg_srmse}
\end{table}


## 2.2. Average Spectral Coherence RMSE: turbulent_radiative_layer_2D

In [22]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}
#band_cols = {"rmse_p_low": "Low", "rmse_p_mid": "Mid", "rmse_p_high": "High"}
#band_cols = {"rmse_lp_low": "Low", "rmse_lp_mid": "Mid", "rmse_lp_high": "High"}

PDE = 'turbulent_radiative_layer_2D'

table = stats_df[stats_df["dataset"] == PDE][["model", "horizon"] + list(band_cols.keys())]

# Melt bands into long format
table = table.melt(
    id_vars=["model", "horizon"],
    value_vars=list(band_cols.keys()),
    var_name="band",
    value_name="psd_rmse"
)
table["band"] = table["band"].map(band_cols)

# Aggregate
table = table.groupby(["model", "horizon", "band"])["psd_rmse"].agg(aggregate).round(3).reset_index()

# Pivot: rows = model, columns = (band, horizon)
table = table.pivot(index="model", columns=["band", "horizon"], values="psd_rmse")

# Reorder bands
table = table.reindex(columns=["Low", "Mid", "High"], level="band")

table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_28502/3656787083.py:59: RuntimeWarning: Mean of empty slice
  return np.nanmean(x)


## 2.3. Average Spectral Coherence RMSE: Rayleigh-Bénard

In [23]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}
#band_cols = {"rmse_p_low": "Low", "rmse_p_mid": "Mid", "rmse_p_high": "High"}
#band_cols = {"rmse_lp_low": "Low", "rmse_lp_mid": "Mid", "rmse_lp_high": "High"}

PDE = 'rayleigh_benard'

table = stats_df[stats_df["dataset"] == PDE][["model", "horizon"] + list(band_cols.keys())]

# Melt bands into long format
table = table.melt(
    id_vars=["model", "horizon"],
    value_vars=list(band_cols.keys()),
    var_name="band",
    value_name="psd_rmse"
)
table["band"] = table["band"].map(band_cols)

# Aggregate
table = table.groupby(["model", "horizon", "band"])["psd_rmse"].agg(aggregate).round(3).reset_index()

# Pivot: rows = model, columns = (band, horizon)
table = table.pivot(index="model", columns=["band", "horizon"], values="psd_rmse")

# Reorder bands
table = table.reindex(columns=["Low", "Mid", "High"], level="band")

table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")

## 2.4. Average Spectral Coherence RMSE: Active-Matter

In [24]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}
#band_cols = {"rmse_p_low": "Low", "rmse_p_mid": "Mid", "rmse_p_high": "High"}
#band_cols = {"rmse_lp_low": "Low", "rmse_lp_mid": "Mid", "rmse_lp_high": "High"}

PDE = 'active_matter'

table = stats_df[stats_df["dataset"] == PDE][["model", "horizon"] + list(band_cols.keys())]

# Melt bands into long format
table = table.melt(
    id_vars=["model", "horizon"],
    value_vars=list(band_cols.keys()),
    var_name="band",
    value_name="psd_rmse"
)
table["band"] = table["band"].map(band_cols)

# Aggregate
table = table.groupby(["model", "horizon", "band"])["psd_rmse"].agg(aggregate).round(3).reset_index()

# Pivot: rows = model, columns = (band, horizon)
table = table.pivot(index="model", columns=["band", "horizon"], values="psd_rmse")

# Reorder bands
table = table.reindex(columns=["Low", "Mid", "High"], level="band")

table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")

## Ablation: Spectral Coherence RMSE for different tcool values

In [25]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}

PDE = 'turbulent_radiative_layer_2D'

table = stats_df[stats_df["dataset"] == PDE][["model"] + constants[PDE] + list(band_cols.keys())]

table = table[table['tcool'] <= 0.030000]

# Melt bands into long format
table = table.melt(
    id_vars=["model"] + constants[PDE],
    value_vars=list(band_cols.keys()),
    var_name="band",
    value_name="psd_rmse"
)
table["band"] = table["band"].map(band_cols)

# Aggregate over both rayleigh and prandtl separately

# --- By tcool
table_ra = (
    table.groupby(["model", "band", constants[PDE][0]])["psd_rmse"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["band", constants[PDE][0]], values="psd_rmse")
    .reindex(columns=["Low", "Mid", "High"], level="band")
)

display(table_ra.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))

band,Low,Mid,High
tcool,0.030000,0.030000,0.030000
model,,,
FM_data_space_v4.0,0.139,0.192,0.153
FNO,0.173,0.236,0.279
FourierFlow_32modes_v4.0,0.240,0.494,0.489
TFNO,0.147,0.228,0.293
WDNO,0.205,0.254,0.360
WNO_3scales_db2,0.518,0.547,0.598
WaveletFlow_1scale_db2_v4.0,0.140,0.190,0.138
WaveletFlow_1scale_db4_v4.0,0.131,0.188,0.134


## Ablation: CRPS for different tcool values

In [26]:
PDE = 'turbulent_radiative_layer_2D'

table = (
    stats_df[["model", "crps", "model_type"] + constants[PDE]]
    .dropna(subset=constants[PDE])      # drop rows with missing parameters
    .dropna(subset=["crps"])            # also drop rows where CRPS itself is missing
)

table['tcool'] = table['tcool'].round(2)

def make_config(row):
    return "\n".join(f"{c}={row[c]}" for c in constants[PDE])

table["config"] = table.apply(make_config, axis=1)
table = table[table["model_type"].isin(["ours", "generative"])]

table_cfg = (
    table.groupby(["model", "config"])["crps"]
    .agg(aggregate)
    .round(3)
    .reset_index()
    .pivot(index="model", columns="config", values="crps")
)

display(
    table_cfg.style
    .apply(highlight_min_per_horizon, axis=None)
    .format("{:.3f}")
)

config,tcool=0.03,tcool=0.06,tcool=0.1,tcool=0.18,tcool=0.32,tcool=0.56,tcool=1.0,tcool=1.78,tcool=3.16
model,,,,,,,,,
FM_data_space_v4.0,1.161,1.145,0.909,1.064,0.926,0.799,0.759,0.968,0.551
FourierFlow_32modes_v4.0,1.390,1.411,1.175,1.429,1.268,1.214,1.140,1.323,0.994
WDNO,3.306,3.298,3.195,3.411,3.360,3.415,3.460,3.560,3.431


## Ablation: Spectral Coherence RMSE for different Rayleigh and Prandtl numbers

In [27]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}

PDE = 'rayleigh_benard'

table = stats_df[stats_df["dataset"] == PDE][["model"] + constants[PDE] + list(band_cols.keys())]

# Melt bands into long format
table = table.melt(
    id_vars=["model"] + constants[PDE],
    value_vars=list(band_cols.keys()),
    var_name="band",
    value_name="psd_rmse"
)
table["band"] = table["band"].map(band_cols)

#table = table[table['Rayleigh'] > 20.]

# Aggregate over both rayleigh and prandtl separately

# --- By Rayleigh (aggregates over all Prandtl configs) ---
table_ra = (
    table.groupby(["model", "band", constants[PDE][0]])["psd_rmse"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["band", constants[PDE][0]], values="psd_rmse")
    .reindex(columns=["Low", "Mid", "High"], level="band")
)

# --- By Prandtl (aggregates over all Rayleigh configs) ---
table_pr = (
    table.groupby(["model", "band", constants[PDE][1]])["psd_rmse"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["band", constants[PDE][1]], values="psd_rmse")
    .reindex(columns=["Low", "Mid", "High"], level="band")
)

display(table_ra.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))
display(table_pr.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))

## Ablation: VRMSE for different Rayleigh and Prandtl numbers 

In [28]:
PDE = 'rayleigh_benard'
table = stats_df[["model", "crps", "model_type"] + constants[PDE]].dropna(subset=constants[PDE])  # drop Ra/Pr NaN rows

# Create a combined Ra/Pr label
table["config"] = table.apply(lambda r: f"Ra={r['Rayleigh']}\nPr={r['Prandtl']}", axis=1)

# filter generative models
table = table[table["model_type"].isin(["ours", "generative"])]

# Aggregate per model / config / horizon
table_cfg = (
    table.groupby(["model", "config"])["crps"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["config"], values="crps")
)
display(table_cfg.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))

config,Ra=13.815510749816896 Pr=-2.3025851249694824,Ra=13.815510749816896 Pr=0.0,Ra=13.815510749816896 Pr=2.3025851249694824,Ra=18.42068099975586 Pr=-2.3025851249694824,Ra=18.42068099975586 Pr=0.0,Ra=18.42068099975586 Pr=2.3025851249694824,Ra=23.025850296020508 Pr=-2.3025851249694824,Ra=23.025850296020508 Pr=0.0,Ra=23.025850296020508 Pr=2.3025851249694824
model,,,,,,,,,
FM_data_space_v4.0,0.097,0.050,0.022,0.097,0.065,0.036,0.076,0.048,0.042
FourierFlow_32modes_v4.0,0.119,0.061,0.033,0.137,0.061,0.045,0.085,0.052,0.054
WDNO,0.160,0.108,0.093,0.143,0.114,0.096,0.131,0.114,0.102


## Ablation: Zeta and Alpha

In [29]:
band_cols = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}

PDE = 'active_matter'

table = stats_df[(stats_df["dataset"] == PDE) & 
                 #(stats_df['step'] <= 32) &
                 (stats_df['model_type'] == 'generative')][["model"] + constants[PDE] + list(band_cols.keys())]

# Melt bands into long format
table = table.melt(
    id_vars=["model"] + constants[PDE],
    value_vars=list(band_cols.keys()),
    var_name="band",
    value_name="psd_rmse"
)
table["band"] = table["band"].map(band_cols)

# Aggregate over both rayleigh and prandtl separately

# --- By Rayleigh (aggregates over all Prandtl configs) ---
table_ra = (
    table.groupby(["model", "band", constants[PDE][0]])["psd_rmse"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["band", constants[PDE][0]], values="psd_rmse")
    .reindex(columns=["Low", "Mid", "High"], level="band")
)

# --- By Prandtl (aggregates over all Rayleigh configs) ---
table_pr = (
    table.groupby(["model", "band", constants[PDE][1]])["psd_rmse"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["band", constants[PDE][1]], values="psd_rmse")
    .reindex(columns=["Low", "Mid", "High"], level="band")
)

display(table_ra.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))
display(table_pr.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))

In [30]:
PDE = 'active_matter'

table = stats_df[["model", "crps", "model_type"] + constants[PDE]].dropna(subset=constants[PDE])

table = table[table['alpha'].isin([-3, -4, -5])]

# Create a combined Ra/Pr label
table["config"] = table.apply(lambda r: f"alpha={r['alpha']}\nzeta={r['zeta']}", axis=1)

# filter generative models
table = table[table["model_type"].isin(["ours", "generative"])]

# Aggregate per model / config / horizon
table_cfg = (
    table.groupby(["model", "config"])["crps"]
    .agg(aggregate).round(3).reset_index()
    .pivot(index="model", columns=["config"], values="crps")
)
display(table_cfg.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}"))

config,alpha=-3.0 zeta=11.0,alpha=-3.0 zeta=17.0,alpha=-3.0 zeta=5.0,alpha=-4.0 zeta=11.0,alpha=-4.0 zeta=17.0,alpha=-5.0 zeta=5.0
model,,,,,,
FM_data_space_v4.0,0.180,0.160,0.168,0.207,0.211,0.272
FourierFlow_32modes_v4.0,0.237,0.224,0.234,0.267,0.253,0.318
WDNO,0.264,0.261,0.276,0.318,0.306,0.382


# 3. Probabilistic Scores

## 3.1. Wasserstein Distance x PDE x Model (View 1)

In [31]:
table = stats_df[stats_df["model_type"].isin(['generative'])][["pde", "model", "horizon", "w1"]]

#table = stats_df[["pde", "model", "horizon"] + list(band_cols.keys())]

table = (
    table.groupby(["pde", "model", "horizon"])
    .agg(aggregate)
    .round(3)
    .reset_index()
    #.rename(columns=band_cols)
    .groupby(["pde", "model"])['w1']
    .mean()
    .round(3)
    .reset_index()
    .pivot(index=["pde", "model"], columns=[], values=['w1'])
)

#table.style.apply(highlight_min_per_horizon, axis=None).format("{:.3f}")
table.style.apply(highlight_min_per_pde, axis=None).format("{:.3f}")
# print(table.to_latex(float_format="%.3f"))

## 3.2. Wasserstein Distance x PDE x Model (View 2)

In [32]:
def highlight_min_per_pde(df):
    """Bold the minimum, underline second minimum, per PDE column."""
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for col in df.columns:  # each column is a PDE
        sorted_vals = df[col].dropna().sort_values()
        best   = sorted_vals.iloc[0] if len(sorted_vals) > 0 else None
        second = sorted_vals.iloc[1] if len(sorted_vals) > 1 else None
        styles[col] = df[col].apply(
            lambda v: "font-weight: bold; color: green"              if v == best   else
                      "text-decoration: underline; color: steelblue" if v == second else ""
        )
    return styles

table = (
    stats_df[stats_df["model_type"].isin(['generative'])][["pde", "model", "w1"]]
    .groupby(["pde", "model"])
    .agg(aggregate)
    .round(3)
    .reset_index()
    .pivot(index="model", columns="pde", values="w1")
)

table.style.apply(highlight_min_per_pde, axis=None).format("{:.3f}")

pde,active_matter,rayleigh_benard,turbulent_radiative_layer_2D
model,,,
FM_data_space_v4.0,0.033,0.018,0.504
FourierFlow_32modes_v4.0,0.105,0.046,1.206
WDNO,0.178,0.110,3.530
